In [0]:
# =============================================================================
# jobs/11_flood_warnings_ingest.py
#
# Notebook:   11_flood_warnings_ingest.py
# Pipeline:   FGS / EA Real-Time Data Pipeline
# Author:     Jon Payne, Environment Agency
# Cadence:    Called by master_polling -- runs every 30 minutes
#
# PURPOSE
# -------
# Fetches the current state of all EA flood warnings and alerts from the
# EA Real-Time Flood Monitoring API and writes to two Delta tables:
#
#   1. flood_warnings_current
#      One row per active flood area. Overwritten on every run.
#      Used by the live dashboard to show what is active right now.
#
#   2. flood_warnings_log
#      Append-only event log. A new row is written whenever a warning's
#      severity or message changes, when a new warning is raised, or when
#      a warning disappears from the feed entirely (CLOSED event).
#      Used for verification analysis and operational audit.
#
# WHY TWO TABLES?
# ---------------
# The API is a snapshot of current state only. It does not provide history.
# By comparing each inbound snapshot against the last known state we can
# reconstruct the full lifecycle of every warning. The log table is the
# result of that comparison. Without it, we lose the event history every
# time the current-state table is overwritten.
#
# SEVERITY LEVELS
# ---------------
# 1 = Severe Flood Warning   (Danger to life)
# 2 = Flood Warning          (Flooding expected, immediate action required)
# 3 = Flood Alert            (Flooding possible, be prepared)
# 4 = Warning no Longer in Force
#
# Level 4 records MUST be stored. They mark the close of an event and remain
# in the feed for approximately 24 hours after a warning ends. If a poll
# cycle misses the level 4 window, the CLOSED event written by this notebook
# is the fallback record of when the warning left the feed.
#
# CHANGE DETECTION
# ----------------
# On each run, this notebook:
#   1. Fetches all records from the API (severity 1-4)
#   2. Loads the current state table into memory
#   3. Compares the two to find:
#      - NEW:              flood_area_id present in API but not in current table
#      - SEVERITY_CHANGE:  same flood_area_id, different severity_level
#      - MESSAGE_CHANGE:   same flood_area_id, same severity, different message
#      - CLOSED:           flood_area_id present in current table but gone from API
#   4. Appends changed and new records to the log table
#   5. Overwrites the current state table with the fresh API snapshot
#
# Note: level 4 records in the current table are treated the same as levels
# 1-3. A level 4 record that then disappears from the feed gets a CLOSED event.
#
# EXIT VALUES (read by master_polling)
# ------------------------------------
# "success"    -- completed without error
# "no_change"  -- API returned data but nothing had changed since last run
# Any exception causes the notebook to fail with an error traceback.
#
# DEPENDENCIES
# ------------
# - prd_dash_lab.flood_forecasting_unrestricted schema must exist
# - EA flood monitoring API must be reachable
# - No secret/auth required -- this is an open API under OGL
# =============================================================================

# =============================================================================
# IMPORTS
# =============================================================================

import requests                          # HTTP requests to the EA API
import json                              # Parse API response
from datetime import datetime, timezone  # Timestamps for log rows
import uuid                              # Unique IDs for log rows
import pandas as pd                      # Local dataframe manipulation
from pyspark.sql import SparkSession     # Write to Delta tables
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, BooleanType, TimestampType
)
from pyspark.sql.functions import col, lit

# =============================================================================
# CONFIGURATION
# =============================================================================

# Unity Catalog path -- all tables live in this schema
CATALOG    = "prd_dash_lab"
SCHEMA     = "flood_forecasting_unrestricted"
DB_PATH    = f"{CATALOG}.{SCHEMA}"

# Table names
TABLE_CURRENT = f"{DB_PATH}.flood_warnings_current"
TABLE_LOG     = f"{DB_PATH}.flood_warnings_log"

# EA Flood Monitoring API endpoint -- returns all current warnings and alerts
# No authentication required. OGL licensed.
# min-severity=4 returns everything including "Warning no Longer in Force"
# so we capture the full picture and never miss a level-4 close event.
API_URL = "https://environment.data.gov.uk/flood-monitoring/id/floods?min-severity=4"

# Request timeout in seconds -- be conservative; EA API can be slow
REQUEST_TIMEOUT = 30


In [0]:
# =============================================================================
# SCHEMA DEFINITIONS
# =============================================================================
# Explicit schemas prevent Spark from inferring types incorrectly when
# the API returns an empty response or a field is absent from some records.

schema_current = StructType([
    StructField("flood_area_id",          StringType(),    nullable=False),
    StructField("severity_level",         IntegerType(),   nullable=True),
    StructField("severity_label",         StringType(),    nullable=True),
    StructField("is_tidal",               BooleanType(),   nullable=True),
    StructField("message",                StringType(),    nullable=True),
    StructField("time_raised",            TimestampType(), nullable=True),
    StructField("time_message_changed",   TimestampType(), nullable=True),
    StructField("time_severity_changed",  TimestampType(), nullable=True),
    StructField("description",            StringType(),    nullable=True),
    StructField("ea_area_name",           StringType(),    nullable=True),
    StructField("river_or_sea",           StringType(),    nullable=True),
    StructField("county",                 StringType(),    nullable=True),
    StructField("ingested_at",            TimestampType(), nullable=False),
])

schema_log = StructType([
    StructField("log_id",                 StringType(),    nullable=False),
    StructField("flood_area_id",          StringType(),    nullable=False),
    StructField("severity_level",         IntegerType(),   nullable=True),
    StructField("severity_label",         StringType(),    nullable=True),
    StructField("is_tidal",               BooleanType(),   nullable=True),
    StructField("message",                StringType(),    nullable=True),
    StructField("time_raised",            TimestampType(), nullable=True),
    StructField("time_message_changed",   TimestampType(), nullable=True),
    StructField("time_severity_changed",  TimestampType(), nullable=True),
    StructField("description",            StringType(),    nullable=True),
    StructField("ea_area_name",           StringType(),    nullable=True),
    StructField("river_or_sea",           StringType(),    nullable=True),
    StructField("county",                 StringType(),    nullable=True),
    StructField("ingested_at",            TimestampType(), nullable=False),
    StructField("change_type",            StringType(),    nullable=False),
])


In [0]:
# =============================================================================
# HELPER: PARSE TIMESTAMP
# =============================================================================
# The API returns timestamps as ISO 8601 strings, e.g. "2024-01-15T09:30:00".
# Some records omit trailing seconds or timezone info. This function handles
# the common variants and returns a Python datetime (UTC, timezone-aware),
# or None if the string is absent or unparseable.

def parse_timestamp(ts_str):
    """
    Parse an ISO 8601 timestamp string from the EA API.
    Returns a timezone-aware datetime (UTC) or None.
    """
    if not ts_str:
        return None
    # Try common formats in order of likelihood
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%dT%H:%M", "%Y-%m-%d"):
        try:
            dt = datetime.strptime(ts_str, fmt)
            # Treat as UTC (the EA API does not include a timezone suffix)
            return dt.replace(tzinfo=timezone.utc)
        except ValueError:
            continue
    # Log the unparseable value and return None rather than crashing
    print(f"  WARNING: Could not parse timestamp: {ts_str!r}")
    return None

# =============================================================================
# HELPER: PARSE API RECORD
# =============================================================================
# Converts a single JSON object from the API response into a flat Python dict
# matching the schema of flood_warnings_current and flood_warnings_log.

def parse_api_record(item, ingested_at):
    """
    Parse one item from the EA floods API response into a flat dict.

    Args:
        item        -- dict from the API 'items' array
        ingested_at -- datetime: when this poll run started

    Returns:
        dict with all fields for the current/log tables
    """
    # The nested floodArea object contains county and riverOrSea.
    # It may be absent in some records, so default to an empty dict.
    flood_area = item.get("floodArea", {})

    return {
        "flood_area_id":          item.get("floodAreaID"),
        "severity_level":         item.get("severityLevel"),
        "severity_label":         item.get("severity"),
        "is_tidal":               item.get("isTidal"),
        "message":                item.get("message"),          # may be absent
        "time_raised":            parse_timestamp(item.get("timeRaised")),
        "time_message_changed":   parse_timestamp(item.get("timeMessageChanged")),
        "time_severity_changed":  parse_timestamp(item.get("timeSeverityChanged")),
        "description":            item.get("description"),
        "ea_area_name":           item.get("eaAreaName"),
        "river_or_sea":           flood_area.get("riverOrSea"),
        "county":                 flood_area.get("county"),
        "ingested_at":            ingested_at,
    }

# =============================================================================
# HELPER: CREATE TABLES IF THEY DO NOT EXIST
# =============================================================================
# On first run, neither table exists. We create them here with the correct
# schema rather than relying on Spark to infer it from the first write.
# On subsequent runs this block is a no-op.

def ensure_tables_exist(spark):
    """
    Create flood_warnings_current and flood_warnings_log if absent.
    Uses Delta format. Does nothing if the tables already exist.
    """
    print("  Checking tables exist...")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TABLE_CURRENT} (
            flood_area_id          STRING    NOT NULL,
            severity_level         INT,
            severity_label         STRING,
            is_tidal               BOOLEAN,
            message                STRING,
            time_raised            TIMESTAMP,
            time_message_changed   TIMESTAMP,
            time_severity_changed  TIMESTAMP,
            description            STRING,
            ea_area_name           STRING,
            river_or_sea           STRING,
            county                 STRING,
            ingested_at            TIMESTAMP NOT NULL
        )
        USING DELTA
        COMMENT 'Current state of all EA flood warnings and alerts. Overwritten on each poll run. One row per active flood area.'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TABLE_LOG} (
            log_id                 STRING    NOT NULL,
            flood_area_id          STRING    NOT NULL,
            severity_level         INT,
            severity_label         STRING,
            is_tidal               BOOLEAN,
            message                STRING,
            time_raised            TIMESTAMP,
            time_message_changed   TIMESTAMP,
            time_severity_changed  TIMESTAMP,
            description            STRING,
            ea_area_name           STRING,
            river_or_sea           STRING,
            county                 STRING,
            ingested_at            TIMESTAMP NOT NULL,
            change_type            STRING    NOT NULL
        )
        USING DELTA
        COMMENT 'Append-only event log of EA flood warning state changes. One row per change event per flood area. change_type: NEW | SEVERITY_CHANGE | MESSAGE_CHANGE | CLOSED.'
    """)

    print("  Tables confirmed.")


In [0]:
# =============================================================================
# STEP 1: FETCH FROM API
# =============================================================================

print("=" * 60)
print("STEP 1: Fetch flood warnings from EA API")
print("=" * 60)

ingested_at = datetime.now(timezone.utc)

try:
    response = requests.get(API_URL, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()   # Raises an exception for HTTP 4xx/5xx
except requests.exceptions.Timeout:
    raise Exception(
        f"EA flood warnings API timed out after {REQUEST_TIMEOUT}s. "
        "The API is sometimes slow. Increase REQUEST_TIMEOUT if this recurs."
    )
except requests.exceptions.RequestException as e:
    raise Exception(f"EA flood warnings API request failed: {e}")

data = response.json()
items = data.get("items", [])

print(f"  API returned {len(items)} records")

if not items:
    # A completely empty response is unusual but valid (e.g. quiet summer).
    # Write an empty current state table and exit cleanly.
    print("  No active warnings or alerts. Updating current state to empty.")
    spark = SparkSession.builder.getOrCreate()
    ensure_tables_exist(spark)
    empty_df = spark.createDataFrame([], schema=schema_current)
    (empty_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(TABLE_CURRENT))
    print("  Current state table cleared.")
    dbutils.notebook.exit("no_change")



In [0]:
# =============================================================================
# STEP 2: PARSE API RESPONSE INTO A LOCAL DATAFRAME
# =============================================================================

print("=" * 60)
print("STEP 2: Parse API response")
print("=" * 60)

# Build a list of flat dicts from the API records
parsed_records = []
skipped = 0

for item in items:
    area_id = item.get("floodAreaID")
    if not area_id:
        # Occasionally a record has no floodAreaID -- skip it
        skipped += 1
        continue
    parsed_records.append(parse_api_record(item, ingested_at))

print(f"  Parsed {len(parsed_records)} records ({skipped} skipped -- no floodAreaID)")

# Convert to a pandas DataFrame for change detection logic.
# The current state table may be large during a flood event (hundreds of areas)
# but is always small enough to fit in driver memory.
api_df = pd.DataFrame(parsed_records)


In [0]:
# =============================================================================
# STEP 3: LOAD CURRENT STATE FROM DELTA
# =============================================================================

print("=" * 60)
print("STEP 3: Load current state from Delta")
print("=" * 60)

spark = SparkSession.builder.getOrCreate()
ensure_tables_exist(spark)

# Read the existing current state table into a pandas DataFrame.
# If the table is empty (e.g. first run or all warnings cleared last run)
# this produces an empty DataFrame with the correct columns.
current_spark_df = spark.table(TABLE_CURRENT)
current_pd = current_spark_df.toPandas()

print(f"  Current state table has {len(current_pd)} rows")

# Index the current state by flood_area_id for fast lookup
current_dict = {}
if len(current_pd) > 0:
    for _, row in current_pd.iterrows():
        current_dict[row["flood_area_id"]] = row


In [0]:
# =============================================================================
# STEP 4: CHANGE DETECTION
# =============================================================================
# Compare the fresh API snapshot against the last known state.
# Classify each change and build rows to append to the log table.

print("=" * 60)
print("STEP 4: Detect changes")
print("=" * 60)

log_rows = []

# --- Check each API record against current state ---
for _, api_row in api_df.iterrows():
    area_id = api_row["flood_area_id"]

    if area_id not in current_dict:
        # This flood area is not in the current state table.
        # It is either brand new, or it disappeared and has come back.
        # Either way, treat it as NEW.
        log_row = api_row.to_dict()
        log_row["log_id"]     = str(uuid.uuid4())
        log_row["change_type"] = "NEW"
        log_rows.append(log_row)
        print(f"    NEW: {area_id} (severity {api_row['severity_level']})")

    else:
        # The area was in the last snapshot. Check for changes.
        prev = current_dict[area_id]
        changes = []

        if api_row["severity_level"] != prev["severity_level"]:
            changes.append("SEVERITY_CHANGE")

        # Message change: compare stripped text, treating None as empty string
        api_msg  = (api_row["message"]  or "").strip()
        prev_msg = (prev["message"]     or "").strip()
        if api_msg != prev_msg and "SEVERITY_CHANGE" not in changes:
            # Only record a MESSAGE_CHANGE if it is not already covered by
            # a severity change (which always implies a new message anyway)
            changes.append("MESSAGE_CHANGE")

        if changes:
            # Determine the primary change type.
            # If both severity and message changed, SEVERITY_CHANGE takes
            # precedence as it is the more significant event.
            change_type = changes[0]
            log_row = api_row.to_dict()
            log_row["log_id"]     = str(uuid.uuid4())
            log_row["change_type"] = change_type
            log_rows.append(log_row)
            print(f"    {change_type}: {area_id} "
                  f"(severity {prev['severity_level']} -> {api_row['severity_level']})")

# --- Check for areas that have disappeared from the API ---
# These are flood areas in the current state table that are not in the
# fresh API response. The warning has been fully removed.
#
# This is distinct from level 4 ("Warning no Longer in Force") which IS
# still present in the API feed. A CLOSED event means the record is gone.
#
# Note: level 4 records that disappear get a CLOSED event here. This is
# the correct behaviour -- they transition from level 4 to fully absent.

api_ids = set(api_df["flood_area_id"].tolist())

for area_id, prev_row in current_dict.items():
    if area_id not in api_ids:
        # Build a CLOSED log row from the last known state, not from the API
        # (there is nothing to read from the API for a missing record).
        log_row = dict(prev_row)
        log_row["log_id"]      = str(uuid.uuid4())
        log_row["ingested_at"] = ingested_at   # Use the current run's timestamp
        log_row["change_type"] = "CLOSED"
        log_rows.append(log_row)
        print(f"    CLOSED: {area_id} (was severity {prev_row['severity_level']})")

print(f"  Change detection complete: {len(log_rows)} events to log")


In [0]:
# =============================================================================
# STEP 5: WRITE LOG ROWS
# =============================================================================

print("=" * 60)
print("STEP 5: Write to log table")
print("=" * 60)

if log_rows:
    log_df_pd = pd.DataFrame(log_rows)

    # Ensure all expected columns are present even if some records were
    # missing optional fields (e.g. 'message' is optional in the API)
    for col_name in [f.name for f in schema_log.fields]:
        if col_name not in log_df_pd.columns:
            log_df_pd[col_name] = None

    # Reorder columns to match the schema
    log_df_pd = log_df_pd[[f.name for f in schema_log.fields]]

    log_spark_df = spark.createDataFrame(log_df_pd, schema=schema_log)

    (log_spark_df.write
        .format("delta")
        .mode("append")           # APPEND -- never overwrite the log
        .saveAsTable(TABLE_LOG))

    print(f"  Appended {len(log_rows)} rows to {TABLE_LOG}")
else:
    print("  No changes detected. Nothing to append to log.")


In [0]:
# =============================================================================
# STEP 6: OVERWRITE CURRENT STATE TABLE
# =============================================================================

print("=" * 60)
print("STEP 6: Overwrite current state table")
print("=" * 60)

# Build the current state DataFrame from the fresh API snapshot
for col_name in [f.name for f in schema_current.fields]:
    if col_name not in api_df.columns:
        api_df[col_name] = None

api_df_ordered = api_df[[f.name for f in schema_current.fields]]
current_spark_new = spark.createDataFrame(api_df_ordered, schema=schema_current)

(current_spark_new.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "false")   # Schema must not change at runtime
    .saveAsTable(TABLE_CURRENT))

print(f"  Wrote {len(api_df)} rows to {TABLE_CURRENT}")



In [0]:
# =============================================================================
# STEP 7: SUMMARY
# =============================================================================

severity_counts = api_df.groupby("severity_level").size().to_dict()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Active warnings/alerts in feed: {len(api_df)}")
for level in sorted(severity_counts.keys()):
    labels = {1: "Severe", 2: "Warning", 3: "Alert", 4: "No longer in force"}
    print(f"    Level {level} ({labels.get(level, '?')}): {severity_counts[level]}")
print(f"  Log events written this run:    {len(log_rows)}")
print("=" * 60)


In [0]:
# =============================================================================
# EXIT
# =============================================================================
# Return "success" to master_polling regardless of whether there were changes.
# The master does not branch on the return value of this job -- it always runs.
# "no_change" is only returned above if the API response was entirely empty.

if log_rows:
    dbutils.notebook.exit("success")
else:
    dbutils.notebook.exit("no_change")
